In [1]:
# ============================================================
# IMPORTS
# ============================================================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import csv, time, json, os


# ============================================================
# CONFIG
# ============================================================
CONFIG_PATH = "ConfigWBBR.json"
INPUT_CSV  = "entrada_referencias.csv"
OUTPUT_CSV = "salida_sucesores.csv"
DELAY = 0.6

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

USER = cfg["USER"]
PWD  = cfg["PWD"]
LOGIN_URL = cfg["LOGIN_URL"]
CSPS_URL  = cfg["CSPS_URL"]


# ============================================================
# BROWSER
# ============================================================
opts = Options()
opts.add_argument("--start-maximized")
driver = webdriver.Chrome(options=opts)
W = WebDriverWait(driver, 25)


# ============================================================
# UTILS
# ============================================================
def ensure_detail():
    driver.switch_to.default_content()
    W.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "detail")))

def open_consulta():
    driver.switch_to.default_content()
    W.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "menu")))
    driver.find_element(By.ID, "folderIcon9").click()
    driver.find_element(By.XPATH, "//a[@title='Consulta General de Referencia']").click()
    ensure_detail()

def get_sucesores_max_2():
    sucesores = []
    try:
        rows = W.until(EC.presence_of_all_elements_located((
            By.XPATH,
            "//table[@id='table1']//tr[td and position()>1]"
        )))
        for r in rows:
            val = r.find_element(By.XPATH, "./td[3]").text.replace("\xa0","").strip()
            if val:
                sucesores.append(val)
            if len(sucesores) == 2:
                break
    except TimeoutException:
        pass

    while len(sucesores) < 2:
        sucesores.append("")

    return sucesores[0], sucesores[1]


# ============================================================
# LOGIN
# ============================================================
driver.get(LOGIN_URL)
W.until(EC.presence_of_element_located((By.ID, "userID"))).send_keys(USER)
driver.find_element(By.ID, "password").send_keys(PWD)
driver.find_element(By.ID, "btn-submit").click()

W.until(EC.url_contains("my.dlrportal.com"))

driver.switch_to.new_window("tab")
driver.get(CSPS_URL)
W.until(EC.presence_of_element_located((By.TAG_NAME, "frameset")))

open_consulta()


# ============================================================
# CSV INPUT
# ============================================================
with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    referencias = [r["referencia"].strip() for r in reader if r["referencia"].strip()]


# ============================================================
# CSV OUTPUT
# ============================================================
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["Referencia", "Sucesor_1", "Sucesor_2"])

    for i, ref in enumerate(referencias, 1):
        try:
            ensure_detail()
            inp = W.until(EC.presence_of_element_located((By.ID, "partNr")))
            inp.clear()
            inp.send_keys(ref)
            inp.send_keys(Keys.ENTER)

            # Esperar sección Sustituciones
            W.until(EC.presence_of_element_located((
                By.XPATH, "//th[contains(.,'Sustituciones')]"
            )))

            s1, s2 = get_sucesores_max_2()
            writer.writerow([ref, s1, s2])
            print(f"[{i}/{len(referencias)}] OK {ref} → {s1} {s2}")

        except Exception as e:
            writer.writerow([ref, "", ""])
            print(f"[{i}/{len(referencias)}] ERROR {ref}")

        time.sleep(DELAY)


# ============================================================
# END
# ============================================================
input("ENTER para cerrar...")
driver.quit()


[1/118] OK 5184118 → 5177709 5184117
[2/118] OK 5191451 → 47492120 
[3/118] OK 87446628 →  
[4/118] OK 11136452 →  
[5/118] OK 11141344 →  
[6/118] ERROR 87269973
[7/118] OK 431868A →  
[8/118] OK 87364602 → 1542831C1 
[9/118] ERROR 5801385949
[10/118] OK 86511440 → 8605075 
[11/118] OK 86998471 → J069053 
[12/118] OK 87381702 →  
[13/118] OK 87381699 →  
[14/118] OK 47813215 → 47556518 
[15/118] OK 84168525 → 47517730 
[16/118] OK 11125568 →  
[17/118] OK 87472289 → 91894003 
[18/118] OK 518806 →  
[19/118] OK 513889 →  
[20/118] OK 521722 →  
[21/118] OK 232170 →  
[22/118] OK 51523932 → 92135909 
[23/118] OK 8510372 →  
[24/118] OK 51574596 → 90471591 
[25/118] OK 11248624 →  
[26/118] OK 47638747 →  
[27/118] OK 90489963 →  
[28/118] ERROR 51527408
[29/118] OK 11137579 →  
[30/118] OK 51524131 →  
[31/118] OK 11060703 →  
[32/118] OK 8100148 →  
[33/118] OK 11017304 →  
[34/118] OK 87042818 →  
[35/118] OK 87687427 →  
[36/118] OK 92066962 →  
[37/118] OK 91809425 →  
[38/118] OK 4

In [2]:
# ============================================================
# IMPORTS
# ============================================================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv, time, json


# ============================================================
# CONFIG
# ============================================================
CONFIG_PATH = "ConfigWBBR.json"
INPUT_CSV  = "entrada_referencias.csv"
OUTPUT_CSV = "salida_sucesores.csv"
DELAY = 0.6

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

USER = cfg["USER"]
PWD  = cfg["PWD"]
LOGIN_URL = cfg["LOGIN_URL"]
CSPS_URL  = cfg["CSPS_URL"]


# ============================================================
# BROWSER
# ============================================================
opts = Options()
opts.add_argument("--start-maximized")
driver = webdriver.Chrome(options=opts)
W = WebDriverWait(driver, 25)


# ============================================================
# UTILS
# ============================================================
def ensure_detail():
    driver.switch_to.default_content()
    W.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "detail")))

def open_consulta():
    driver.switch_to.default_content()
    W.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "menu")))
    driver.find_element(By.ID, "folderIcon9").click()
    driver.find_element(By.XPATH, "//a[@title='Consulta General de Referencia']").click()
    ensure_detail()

def get_sucesores_max_2():
    """
    NUNCA espera filas.
    Si la tabla está vacía → devuelve inmediatamente.
    """
    sucesores = []

    rows = driver.find_elements(
        By.XPATH,
        "//table[@id='table1']//tr[position()>1]"
    )

    if not rows:
        return "", ""

    for r in rows:
        try:
            tds = r.find_elements(By.TAG_NAME, "td")
            if len(tds) >= 3:
                val = tds[2].text.replace("\xa0", "").strip()
                if val:
                    sucesores.append(val)
        except Exception:
            pass

        if len(sucesores) == 2:
            break

    while len(sucesores) < 2:
        sucesores.append("")

    return sucesores[0], sucesores[1]


# ============================================================
# LOGIN
# ============================================================
driver.get(LOGIN_URL)
W.until(EC.presence_of_element_located((By.ID, "userID"))).send_keys(USER)
driver.find_element(By.ID, "password").send_keys(PWD)
driver.find_element(By.ID, "btn-submit").click()

W.until(EC.url_contains("my.dlrportal.com"))

driver.switch_to.new_window("tab")
driver.get(CSPS_URL)
W.until(EC.presence_of_element_located((By.TAG_NAME, "frameset")))

open_consulta()


# ============================================================
# INPUT CSV
# ============================================================
with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    referencias = [r["referencia"].strip() for r in reader if r["referencia"].strip()]


# ============================================================
# OUTPUT CSV
# ============================================================
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["Referencia", "Sucesor_1", "Sucesor_2"])

    for i, ref in enumerate(referencias, 1):
        try:
            ensure_detail()

            inp = W.until(EC.presence_of_element_located((By.ID, "partNr")))
            inp.clear()
            inp.send_keys(ref)
            inp.send_keys(Keys.ENTER)

            # Esperar que cargue la página (NO filas)
            W.until(EC.presence_of_element_located((
                By.XPATH, "//table[@id='table1']"
            )))

            s1, s2 = get_sucesores_max_2()
            writer.writerow([ref, s1, s2])
            print(f"[{i}/{len(referencias)}] OK {ref} → {s1} {s2}")

        except Exception:
            writer.writerow([ref, "", ""])
            print(f"[{i}/{len(referencias)}] ERROR {ref}")

        time.sleep(DELAY)


# ============================================================
# END
# ============================================================
input("ENTER para cerrar...")
driver.quit()


[1/101] OK 135294A1 → 47953510 
[2/101] OK 602958R91 → 602957R91 
[3/101] OK 87662557 →  
[4/101] OK 84310960 →  
[5/101] OK 5801574786 →  
[6/101] OK 9833222 →  
[7/101] OK 1347351C2 → 47946917 
[8/101] OK 5801878188 →  
[9/101] OK 5801878191 →  
[10/101] OK 87681210 →  
[11/101] OK 84362299 →  
[12/101] OK 51466254 → 92204614 
[13/101] OK 48068706 →  
[14/101] OK 320033 →  
[15/101] OK 5802055312 →  
[16/101] OK 47490860 →  
[17/101] OK 90433959 →  
[18/101] OK 84362292 →  
[19/101] OK 47411931 →  
[20/101] OK 87728304 →  
[21/101] OK 84071125 →  
[22/101] OK 87748377 →  
[23/101] OK 87283318 →  
[24/101] OK 87434005 →  
[25/101] OK 84582096 →  
[26/101] OK 47775916 →  
[27/101] OK 84446877 →  
[28/101] OK 48175281 →  
[29/101] OK 84479382 →  
[30/101] OK 51676887 →  
[31/101] OK 51429826 →  
[32/101] OK 87396383 →  
[33/101] OK 87526988 →  
[34/101] OK 86638314 →  
[35/101] OK 87011804 →  
[36/101] OK 87314571 →  
[37/101] OK 87314573 →  
[38/101] OK 47798422 →  
[39/101] OK 4807389